In [1]:
# All package imports (run this cell first)
import sys
import subprocess
import json
from pathlib import Path

import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import PeftModel, get_peft_model, LoraConfig, TaskType

## Kernel check (use root .venv)

Run this cell first to confirm the notebook is using the project's root `.venv`.

In [2]:
# Kernel verification
_venv_ok = "My-Crew-Manager" in sys.executable and ".venv" in sys.executable
print(f"Python: {sys.executable}")
print(f"Using root .venv: {'✓ Yes' if _venv_ok else '✗ No – select Kernel → Python (My-Crew-Manager .venv)'}")

Python: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\.venv\Scripts\python.exe
Using root .venv: ✓ Yes


# Model Part 1 - Proposal Summarization and Sprint Planning

From raw project description to structured Part 1 output (summary, roles, features, goals, timeline).

Run from AI folder or project root. This notebook expects an existing Model 1 JSONL dataset in llms/fine_tune/dataset.

## Setup paths

In [3]:
# Resolve AI root
_cwd = Path.cwd()
_ai_root = _cwd if (_cwd / "llms").exists() else (_cwd / "AI" if (_cwd / "AI").exists() else _cwd)
if str(_ai_root) not in sys.path:
    sys.path.insert(0, str(_ai_root))

FINE_TUNE_DIR = _ai_root / "llms" / "fine_tune"
DATASET_DIR = FINE_TUNE_DIR / "dataset"
TOKENIZED_DIR = FINE_TUNE_DIR / "tokenized"
OUTPUT_DIR = FINE_TUNE_DIR / "qwen_model1_overview_lora_1p5b"

print(f"AI root: {_ai_root}")
print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")

AI root: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI
Dataset: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\dataset
Output: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model1_overview_lora_1p5b


## Step 1: Import existing Model 1 dataset

Load the prepared `model1_description_to_part1.jsonl` dataset and preview sample records.

In [6]:
# Import and preview Model 1 dataset
dataset_file = DATASET_DIR / "model1_description_to_part1.jsonl"
if not dataset_file.exists():
    raise FileNotFoundError(
        f"Missing dataset: {dataset_file}. Generate or convert datasets first."
    )

rows = []
for line in dataset_file.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line:
        continue
    rows.append(json.loads(line))

print(f"Loaded {len(rows)} examples from {dataset_file.name}")
print("Model 1 contract:")
print("- Prompt: Proposal / Input text from user")
print("- Response: Title + Summary + Roles + Features + Goals + Timeline")
print("- Excluded from response: Status, Proposal / Input, Backlog")
if rows:
    print("\nSample prompt:")
    print(rows[0]["prompt"][:220] + ("..." if len(rows[0]["prompt"]) > 220 else ""))
    print("\nSample response snippet:")
    print(rows[0]["response"][:280] + ("..." if len(rows[0]["response"]) > 280 else ""))

Loaded 26 examples from model1_description_to_part1.jsonl
Model 1 contract:
- Prompt: Proposal / Input text from user
- Response: Title + Summary + Roles + Features + Goals + Timeline
- Excluded from response: Status, Proposal / Input, Backlog

Sample prompt:
Develop the MyCrewManager platform, a web-based system that enables project managers to assign tasks, track progress, and receive AI-generated recommendations for team coordination and risk mitigation. The project will c...

Sample response snippet:
=== MyCrewManager ===
Summary:
Develop the MyCrewManager platform, a web-based system that enables project managers to assign tasks, track progress, and receive AI-generated recommendations for team coordination and risk mitigation. The project will consist of a Django REST API b...


## Step 2: Prepare tokenized dataset

In [13]:
from llms.fine_tune.prepare_dataset import prepare_model1_dataset

DATASET_FILENAME = "model1_description_to_part1_dualprompt_v3.jsonl"
source_dataset_file = DATASET_DIR / DATASET_FILENAME
tokenized_path = TOKENIZED_DIR / "tokenized_model1_qwen_dualprompt_v3"
MAX_LENGTH = 512
FORCE_REBUILD_TOKENIZED = False

if not source_dataset_file.exists():
    raise FileNotFoundError(f"Missing source dataset: {source_dataset_file}")

source_count = sum(1 for line in source_dataset_file.read_text(encoding="utf-8").splitlines() if line.strip())
print(f"Source examples in JSONL ({DATASET_FILENAME}): {source_count}")

rebuild_needed = FORCE_REBUILD_TOKENIZED or not tokenized_path.exists()
if tokenized_path.exists() and not rebuild_needed:
    dataset = load_from_disk(str(tokenized_path))
    tokenized_count = len(dataset)
    if tokenized_count != source_count:
        print(
            f"Tokenized dataset is stale (tokenized={tokenized_count}, source={source_count}). "
            f"Rebuilding..."
        )
        rebuild_needed = True

if rebuild_needed:
    if tokenized_path.exists():
        import shutil
        shutil.rmtree(tokenized_path)
    dataset = prepare_model1_dataset(
        model_name="qwen",
        max_length=MAX_LENGTH,
        dataset_filename=DATASET_FILENAME,
        output_dir=str(tokenized_path),
    )
    print(f"Rebuilt tokenized dataset at {tokenized_path}")
else:
    print(f"Loaded tokenized dataset from {tokenized_path}")

# 60/20/20 split: train / eval / holdout
split_primary = dataset.train_test_split(test_size=0.4, seed=42)
train_dataset = split_primary["train"]

split_secondary = split_primary["test"].train_test_split(test_size=0.5, seed=42)
eval_dataset = split_secondary["train"]
holdout_dataset = split_secondary["test"]

print(f"Total tokenized examples: {len(dataset)}")
print(f"Train size (60%): {len(train_dataset)}")
print(f"Eval size (20%): {len(eval_dataset)}")
print(f"Holdout size (20%): {len(holdout_dataset)}")
print(f"Max length: {MAX_LENGTH}")
print("\nUsing improved dual-prompt training dataset with EPIC-level Goals and no Status field.")

Source examples in JSONL (model1_description_to_part1_dualprompt_v3.jsonl): 100
Loaded tokenized dataset from c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\tokenized\tokenized_model1_qwen_dualprompt_v3
Total tokenized examples: 100
Train size (60%): 60
Eval size (20%): 20
Holdout size (20%): 20
Max length: 512

Using improved dual-prompt training dataset with EPIC-level Goals and no Status field.


## Step 3: Load model & apply LoRA

In [4]:
MODEL_ID = "Qwen/Qwen2-1.5B-Instruct"
BATCH_SIZE = 2
EPOCHS = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
if torch.cuda.is_available():
    model = model.to("cuda")

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


## Step 4: Train iteratively until target

Train in capped rounds, evaluate after each round, and stop early when quality targets are reached.

In [14]:
import gc
import inspect
import math
import os
import random
import shutil
from pathlib import Path

os.environ.setdefault("TENSORBOARD_LOGGING_DIR", str(OUTPUT_DIR / "logs"))
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

TARGET_EVAL_LOSS = 1.2
TARGET_PERPLEXITY = 3.5
MAX_ROUNDS = 10
MIN_ROUNDS = 1
PATIENCE_ROUNDS = 2
EPOCHS_PER_ROUND = 3
SEEDS = [42, 123]

# Memory-safe effective batch size: 2 (1 x accumulation 2)
TRAIN_BATCH_SIZE = 1
GRAD_ACC_STEPS = 2

TRIALS_DIR = OUTPUT_DIR / "trials"
TRIALS_DIR.mkdir(parents=True, exist_ok=True)


def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clear_cuda_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


training_kwargs_base = {
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACC_STEPS,
    "per_device_eval_batch_size": 1,
    "num_train_epochs": EPOCHS_PER_ROUND,
    "logging_steps": 10,
    "fp16": torch.cuda.is_available(),
    "report_to": "none",
    "save_strategy": "no",
    "dataloader_pin_memory": False,
    "prediction_loss_only": True,
    "torch_empty_cache_steps": 1,
    "disable_tqdm": True,
}
if "evaluation_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs_base["evaluation_strategy"] = "no"
else:
    training_kwargs_base["eval_strategy"] = "no"

trial_summaries = []
global_best = None

for seed in SEEDS:
    print(f"\n{'=' * 80}")
    print(f"Starting trial for seed={seed}")
    print(f"{'=' * 80}")

    set_global_seed(seed)
    clear_cuda_cache()

    trial_dir = TRIALS_DIR / f"seed_{seed}"
    if trial_dir.exists():
        shutil.rmtree(trial_dir)
    trial_dir.mkdir(parents=True, exist_ok=True)

    trial_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        trust_remote_code=True,
    )
    if torch.cuda.is_available():
        trial_model = trial_model.to("cuda")

    trial_model = get_peft_model(trial_model, peft_config)

    trial_kwargs = dict(training_kwargs_base)
    trial_kwargs["output_dir"] = str(trial_dir)
    trial_kwargs["seed"] = seed
    trial_kwargs["data_seed"] = seed

    trial_args = TrainingArguments(**trial_kwargs)

    trainer = Trainer(
        model=trial_model,
        args=trial_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    round_history = []
    best_round = None
    best_checkpoint_dir = None
    rounds_without_improvement = 0
    stop_reason = "max_rounds_reached"

    for round_num in range(1, MAX_ROUNDS + 1):
        print(f"\n--- Seed {seed} | Round {round_num}/{MAX_ROUNDS} ---")
        train_result = trainer.train()

        pred_output = trainer.predict(eval_dataset, metric_key_prefix="eval")
        eval_metrics = pred_output.metrics
        eval_loss = eval_metrics.get("eval_loss") or eval_metrics.get("test_loss")
        perplexity = math.exp(eval_loss) if eval_loss is not None and eval_loss < 20 else float("inf")

        round_checkpoint_dir = trial_dir / f"round_{round_num}"
        round_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(round_checkpoint_dir))

        round_entry = {
            "seed": seed,
            "round": round_num,
            "train_loss": float(train_result.training_loss),
            "eval_loss": float(eval_loss) if eval_loss is not None else None,
            "perplexity": float(perplexity),
            "checkpoint_dir": str(round_checkpoint_dir),
        }
        round_history.append(round_entry)

        print(f"Train loss: {round_entry['train_loss']:.4f}")
        if eval_loss is not None:
            print(f"Eval loss: {eval_loss:.4f}")
        else:
            print("Eval loss: unavailable")
        print(f"Perplexity: {perplexity:.4f}" if math.isfinite(perplexity) else "Perplexity: inf")

        is_better = False
        if eval_loss is not None:
            if best_round is None:
                is_better = True
            else:
                if eval_loss < best_round["eval_loss"]:
                    is_better = True
                elif eval_loss == best_round["eval_loss"] and perplexity < best_round["perplexity"]:
                    is_better = True

        if is_better:
            best_round = round_entry
            best_checkpoint_dir = round_checkpoint_dir
            rounds_without_improvement = 0
            print("Improvement detected: updated best checkpoint.")
        else:
            rounds_without_improvement += 1
            print(f"No improvement. Patience counter: {rounds_without_improvement}/{PATIENCE_ROUNDS}")

        if eval_loss is not None and (eval_loss <= TARGET_EVAL_LOSS or perplexity <= TARGET_PERPLEXITY):
            if round_num >= MIN_ROUNDS:
                stop_reason = "target_reached"
                print(
                    f"Stopping: target reached at round {round_num} "
                    f"(eval_loss={eval_loss:.4f}, perplexity={perplexity:.4f})"
                )
                break

        if round_num >= MIN_ROUNDS and rounds_without_improvement >= PATIENCE_ROUNDS:
            stop_reason = "patience_exhausted"
            print(f"Stopping: no improvement for {PATIENCE_ROUNDS} rounds.")
            break

        clear_cuda_cache()

    if best_checkpoint_dir is None:
        print(f"Seed {seed}: no valid checkpoint selected; skipping holdout evaluation.")
        del trainer, trial_model
        clear_cuda_cache()
        continue

    holdout_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        trust_remote_code=True,
    )
    best_model = PeftModel.from_pretrained(holdout_base, str(best_checkpoint_dir))
    if torch.cuda.is_available():
        best_model = best_model.to("cuda")

    holdout_args = TrainingArguments(
        output_dir=str(trial_dir / "holdout_eval"),
        report_to="none",
        per_device_eval_batch_size=1,
        dataloader_pin_memory=False,
        prediction_loss_only=True,
        disable_tqdm=True,
    )
    holdout_trainer = Trainer(
        model=best_model,
        args=holdout_args,
        eval_dataset=holdout_dataset,
        processing_class=tokenizer,
    )

    holdout_pred = holdout_trainer.predict(holdout_dataset, metric_key_prefix="holdout")
    holdout_metrics = holdout_pred.metrics
    holdout_loss = holdout_metrics.get("holdout_loss") or holdout_metrics.get("test_loss")
    holdout_perplexity = math.exp(holdout_loss) if holdout_loss is not None and holdout_loss < 20 else float("inf")

    trial_summary = {
        "seed": seed,
        "stop_reason": stop_reason,
        "rounds_completed": len(round_history),
        "best_round": best_round,
        "best_checkpoint_dir": str(best_checkpoint_dir),
        "holdout_loss": float(holdout_loss) if holdout_loss is not None else None,
        "holdout_perplexity": float(holdout_perplexity),
    }
    trial_summaries.append(trial_summary)

    print(f"\nSeed {seed} summary")
    print(f"- Stop reason: {stop_reason}")
    print(f"- Best round: {best_round['round']} (eval_loss={best_round['eval_loss']:.4f}, perplexity={best_round['perplexity']:.4f})")
    print(
        f"- Holdout: loss={trial_summary['holdout_loss']:.4f}, "
        f"perplexity={trial_summary['holdout_perplexity']:.4f}"
        if trial_summary["holdout_loss"] is not None
        else "- Holdout: unavailable"
    )

    if global_best is None:
        global_best = trial_summary
    else:
        cur_loss = trial_summary["holdout_loss"]
        best_loss = global_best["holdout_loss"]
        cur_ppl = trial_summary["holdout_perplexity"]
        best_ppl = global_best["holdout_perplexity"]

        if cur_loss is not None and (best_loss is None or cur_loss < best_loss):
            global_best = trial_summary
        elif cur_loss is not None and best_loss is not None and cur_loss == best_loss and cur_ppl < best_ppl:
            global_best = trial_summary

    del holdout_trainer, best_model, holdout_base, trainer, trial_model
    clear_cuda_cache()

if not trial_summaries:
    raise RuntimeError("No successful trial completed. Cannot promote best model.")

print(f"\n{'=' * 80}")
print("TRIAL RESULTS")
print(f"{'=' * 80}")
for t in trial_summaries:
    br = t["best_round"]
    holdout_loss_txt = f"{t['holdout_loss']:.4f}" if t["holdout_loss"] is not None else "NA"
    print(
        f"Seed {t['seed']}: stop={t['stop_reason']}, rounds={t['rounds_completed']}, "
        f"best_round={br['round']} (eval_loss={br['eval_loss']:.4f}, perplexity={br['perplexity']:.4f}), "
        f"holdout_loss={holdout_loss_txt}, holdout_perplexity={t['holdout_perplexity']:.4f}"
    )

print(f"\nGlobal winner seed: {global_best['seed']}")
print(f"Winner checkpoint: {global_best['best_checkpoint_dir']}")

winner_seed = global_best["seed"]
winner_checkpoint_dir = Path(global_best["best_checkpoint_dir"])
winner_holdout_loss = global_best["holdout_loss"]
winner_holdout_perplexity = global_best["holdout_perplexity"]
training_trial_summaries = trial_summaries


Starting trial for seed=42


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



--- Seed 42 | Round 1/10 ---


RuntimeError: CUDA error: CUBLAS_STATUS_INTERNAL_ERROR when calling `cublasGemmEx( handle, opa, opb, m, n, k, alpha_ptr, a, CUDA_R_16F, lda, b, CUDA_R_16F, ldb, beta_ptr, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16F, ldc, compute_type, CUBLAS_GEMM_DEFAULT_TENSOR_OP)`

## Step 4.1: Evaluate model performance

Compute evaluation loss and perplexity on the held-out evaluation split.

In [9]:
import gc

# Evaluate promoted winner checkpoint on eval and holdout splits
if "winner_checkpoint_dir" not in globals():
    raise RuntimeError("Run the training cell first to select a winner checkpoint.")

winner_checkpoint_dir = Path(winner_checkpoint_dir)
if not winner_checkpoint_dir.exists():
    raise RuntimeError(f"Winner checkpoint not found: {winner_checkpoint_dir}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
winner_model = PeftModel.from_pretrained(eval_base, str(winner_checkpoint_dir))
if torch.cuda.is_available():
    winner_model = winner_model.to("cuda")

eval_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "winner_eval"),
    report_to="none",
    per_device_eval_batch_size=1,
    dataloader_pin_memory=False,
    prediction_loss_only=True,
    disable_tqdm=True,
)
eval_trainer = Trainer(
    model=winner_model,
    args=eval_args,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

eval_metrics = eval_trainer.predict(eval_dataset, metric_key_prefix="eval").metrics
eval_loss = eval_metrics.get("eval_loss") or eval_metrics.get("test_loss")

holdout_metrics = eval_trainer.predict(holdout_dataset, metric_key_prefix="holdout").metrics
holdout_loss = holdout_metrics.get("holdout_loss") or holdout_metrics.get("test_loss")

eval_perplexity = math.exp(eval_loss) if eval_loss is not None and eval_loss < 20 else float("inf")
holdout_perplexity = math.exp(holdout_loss) if holdout_loss is not None and holdout_loss < 20 else float("inf")

print("Winner checkpoint evaluation")
print(f"Winner seed: {winner_seed}")
print(f"Winner checkpoint: {winner_checkpoint_dir}")
print(f"Eval loss: {eval_loss:.4f}" if eval_loss is not None else "Eval loss: unavailable")
print(f"Eval perplexity: {eval_perplexity:.4f}" if math.isfinite(eval_perplexity) else "Eval perplexity: inf")
print(f"Holdout loss: {holdout_loss:.4f}" if holdout_loss is not None else "Holdout loss: unavailable")
print(f"Holdout perplexity: {holdout_perplexity:.4f}" if math.isfinite(holdout_perplexity) else "Holdout perplexity: inf")

print("\nAll eval metrics:")
print(eval_metrics)
print("\nAll holdout metrics:")
print(holdout_metrics)

# Release large eval objects before adapter save and inference.
for _name in ("eval_trainer", "winner_model", "eval_base"):
    if _name in globals():
        del globals()[_name]
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("\n[Memory cleanup] Released eval objects and cleared CUDA cache.")

RuntimeError: Run the training cell first to select a winner checkpoint.

## Step 5: Save adapter

In [10]:
import gc
import shutil

# Defensive cleanup for out-of-order reruns before copying artifacts.
for _name in (
    "eval_trainer",
    "winner_model",
    "eval_base",
    "holdout_trainer",
    "best_model",
    "holdout_base",
    "model_infer",
    "base_model",
):
    if _name in globals():
        del globals()[_name]
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

if "winner_checkpoint_dir" not in globals():
    raise RuntimeError("Run training first so a winner checkpoint is selected.")

winner_checkpoint_dir = Path(winner_checkpoint_dir)
if not winner_checkpoint_dir.exists():
    raise RuntimeError(f"Winner checkpoint missing: {winner_checkpoint_dir}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Clean existing adapter artifacts (keep trials folder if present)
for child in OUTPUT_DIR.iterdir():
    if child.name == "trials":
        continue
    if child.is_dir():
        shutil.rmtree(child)
    else:
        child.unlink()

# Copy winner adapter into final output directory
for child in winner_checkpoint_dir.iterdir():
    dst = OUTPUT_DIR / child.name
    if child.is_dir():
        shutil.copytree(child, dst)
    else:
        shutil.copy2(child, dst)

# Ensure tokenizer files are present for inference
tokenizer.save_pretrained(str(OUTPUT_DIR))

print(f"Promoted winner checkpoint from: {winner_checkpoint_dir}")
print(f"Saved final adapter and tokenizer to {OUTPUT_DIR}")
print("Set PEFT_ADAPTER_PATH in AI/.env to use this adapter:")
print("  PEFT_ADAPTER_PATH=llms/fine_tune/qwen_model1_overview_lora_1p5b")

Promoted winner checkpoint from: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model1_overview_lora_1p5b\trials\seed_42\round_1
Saved final adapter and tokenizer to c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model1_overview_lora_1p5b
Set PEFT_ADAPTER_PATH in AI/.env to use this adapter:
  PEFT_ADAPTER_PATH=llms/fine_tune/qwen_model1_overview_lora_1p5b


## Step 6: Quick inference test

In [ ]:
import gc
import json
import os
import re
from datetime import datetime

# Final VRAM cleanup before loading inference models.
for _name in ("eval_trainer", "winner_model", "eval_base", "model", "trainer"):
    if _name in globals():
        del globals()[_name]
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

adapter_rel = os.getenv("PEFT_ADAPTER_PATH", "llms/fine_tune/qwen_model1_overview_lora_1p5b")
adapter_path_obj = Path(adapter_rel)
if adapter_path_obj.is_absolute():
    PROMOTED_ADAPTER_DIR = adapter_path_obj
else:
    PROMOTED_ADAPTER_DIR = (_ai_root / adapter_path_obj).resolve()

if not PROMOTED_ADAPTER_DIR.exists():
    raise FileNotFoundError(f"Promoted adapter not found: {PROMOTED_ADAPTER_DIR}")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
model_infer = PeftModel.from_pretrained(base_model, str(PROMOTED_ADAPTER_DIR))
if torch.cuda.is_available():
    model_infer = model_infer.to("cuda")

model_infer.eval()
for param in model_infer.parameters():
    param.requires_grad = False

# Load few-shot examples from the latest dualprompt_v3 dataset
dualprompt_v3_file = DATASET_DIR / "model1_description_to_part1_dualprompt_v3.jsonl"
few_shot_examples = []
with open(dualprompt_v3_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 2:  # Take first 2 examples (one raw, one guided variant)
            break
        few_shot_examples.append(json.loads(line.strip()))

# Enhanced few-shot prefix with actual good examples
FEWSHOT_CONTEXT = ""
if len(few_shot_examples) >= 2:
    ex1_resp = few_shot_examples[0]["response"]
    FEWSHOT_CONTEXT = f"Example of correct output structure:\n{ex1_resp[:350]}\n\n"

PROMPTS_DIR = _ai_root / "llms" / "prompts"
STRICT_PROMPT_SECTIONS = ["summary", "roles", "features", "goals", "timeline"]
USE_STRICT_PROMPT_RULES = True
MAX_RULE_LINES_PER_SECTION = 6
OUTPUT_END_MARKER = "END_OF_OVERVIEW"
EXTRA_STRICT_RULE_LINES = [
    "Use each required heading exactly once; do not repeat sections.",
    "Do not restart with a new Title or Summary after Timeline.",
    "After Week 4, output END_OF_OVERVIEW and stop.",
    "Output only one final draft; no second attempt or continuation text.",
]


def _load_prompt_template(section):
    prompt_path = PROMPTS_DIR / f"{section}_prompt.txt"
    if not prompt_path.exists():
        return None
    try:
        return prompt_path.read_text(encoding="utf-8")
    except OSError:
        return None


def _extract_section_rules(section, template):
    lines = template.splitlines()
    rules = []
    in_rules_block = False
    _ = section

    for line in lines:
        stripped = line.strip()
        lowered = stripped.lower()

        if not stripped:
            continue
        if stripped.startswith("<<<"):
            continue

        if lowered.startswith("rules:"):
            in_rules_block = True
            continue

        if lowered.startswith("example format:"):
            in_rules_block = False
            continue

        if in_rules_block:
            m = re.match(r"^\d+\.\s*(.+)$", stripped)
            if m:
                rules.append(m.group(1).strip())
            continue

        if section == "summary":
            if "must start with" in lowered and "summary:" in lowered:
                rules.append("Use heading Summary:.")
            if "2" in stripped and "sentence" in lowered:
                rules.append("Keep Summary concise (2-3 sentences).")
            if "overall purpose" in lowered or "success criteria" in lowered:
                rules.append("Summary should cover purpose, scope, team, and success criteria.")

    return rules


def _normalize_rule_for_combined_output(section, rule):
    lowered = rule.lower()

    skip_markers = [
        "exactly two spaces",
        "exactly four spaces",
        "output only the yaml block",
        "example format",
        "<<<",
        "{proposal_text}",
        "{roles}",
        "{additional_roles}",
    ]
    if any(marker in lowered for marker in skip_markers):
        return None

    heading_map = {
        "summary": "Summary",
        "roles": "Roles",
        "features": "Features",
        "goals": "Goals",
        "timeline": "Timeline",
    }

    if "start with" in lowered and "prefix" in lowered:
        return f"Use heading '{heading_map.get(section, section.title())}:' for this section."

    if section == "timeline":
        if "exactly 4 weeks" in lowered or "include exactly 4 weeks" in lowered:
            return "Timeline must include Week 1 through Week 4 only."
        if "exactly 2 tasks" in lowered:
            return "Each timeline week must include exactly two concrete tasks."
        if "no role assignments" in lowered:
            return "Do not include role assignments in the Timeline section."

    if section == "features":
        if "list max 6" in lowered:
            return "List at most six core Features."
        if "under 5 words" in lowered:
            return "Keep each Feature concise (about five words or fewer)."

    if section == "roles":
        if "include exactly these 5 core roles first" in lowered:
            return "Roles should prioritize core delivery roles (Project Manager, Frontend Developer, Backend Developer, Quality Assurance Engineer, UI/UX Designer)."
        if "add up to 3 additional roles" in lowered:
            return "Roles may include up to three additional project-specific roles."
        if "title case" in lowered:
            return "Use Title Case for Roles entries."

    if section == "goals":
        if "high-level functional area" in lowered or "major project goal" in lowered or "epic" in lowered:
            return "Goals should be concise EPIC-level objectives distinct from Features."
        if "create exactly 8 max goals" in lowered:
            return "List no more than eight Goals."

    if section == "summary":
        if "2-3" in lowered or "2-3 sentences" in lowered:
            return "Keep Summary to about 2-3 sentences."
        if "purpose" in lowered and "success criteria" in lowered:
            return "Summary should cover purpose, scope, team composition, and success criteria."

    if "no descriptions" in lowered or "no comments" in lowered:
        return "Avoid meta commentary; output only project content."

    return None


def _build_strict_constraints_block():
    loaded_sections = []
    all_rules = []

    for section in STRICT_PROMPT_SECTIONS:
        template = _load_prompt_template(section)
        if not template:
            continue

        loaded_sections.append(section)
        raw_rules = _extract_section_rules(section, template)

        normalized = []
        for rule in raw_rules:
            mapped = _normalize_rule_for_combined_output(section, rule)
            if mapped and mapped not in normalized:
                normalized.append(mapped)

        all_rules.extend(normalized[:MAX_RULE_LINES_PER_SECTION])

    deduped = []
    seen = set()
    for rule in all_rules:
        if rule not in seen:
            deduped.append(rule)
            seen.add(rule)

    for extra_rule in EXTRA_STRICT_RULE_LINES:
        if extra_rule not in seen:
            deduped.append(extra_rule)
            seen.add(extra_rule)

    if not deduped:
        return "", loaded_sections, []

    block = "Additional strict format guidance (derived from prompt templates):\n"
    block += "\n".join(f"- {rule}" for rule in deduped)
    block += "\n\n"
    return block, loaded_sections, deduped


BASE_GUIDED_PREFIX = (
    "Generate ONLY the following sections in order: "
    "Title, Summary, Roles, Features, Goals, Timeline (Week 1-4).\n"
    "Use each heading once only; do not repeat sections.\n"
    "Do not include Status, Proposal/Input, or Backlog.\n"
    "Goals must be EPIC-level objectives distinct from Features.\n"
    "Timeline must have concrete, distinct tasks for each week and exactly Week 1 through Week 4.\n"
    f"After Week 4, output only '{OUTPUT_END_MARKER}' on its own line and stop.\n\n"
)

strict_constraints_block, strict_loaded_sections, strict_rule_lines = _build_strict_constraints_block()
strict_prompt_integration_active = bool(USE_STRICT_PROMPT_RULES and strict_rule_lines)

if strict_prompt_integration_active:
    GUIDED_PREFIX = BASE_GUIDED_PREFIX + strict_constraints_block
else:
    GUIDED_PREFIX = BASE_GUIDED_PREFIX

# Inference speed controls
QUICK_RUN_MODE = False  # Default fast mode; set False for full validation run
STRATEGIES_TO_RUN = ["B"] if QUICK_RUN_MODE else ["A", "B", "C"]
TOKENS_BY_STRATEGY = {"A": 260, "B": 300, "C": 320}
SHOW_FULL_OUTPUT = True  # Set to False to only show truncated output snippets in quick mode
EXPORT_OUTPUT_TXT = True
EXPORT_DIR = PROMOTED_ADAPTER_DIR / "inference_test_exports"

# Deterministic proposal selection (1-based dataset row numbers).
# Edit this list to choose the exact 5 proposals used for inference tests.
SELECTED_PROPOSAL_ROWS = [6, 7, 8, 9, 10]

if len(SELECTED_PROPOSAL_ROWS) != 5:
    raise ValueError(
        "SELECTED_PROPOSAL_ROWS must contain exactly 5 row numbers "
        f"(received {len(SELECTED_PROPOSAL_ROWS)})."
    )
if not all(isinstance(x, int) for x in SELECTED_PROPOSAL_ROWS):
    raise ValueError("SELECTED_PROPOSAL_ROWS must contain integers only.")
if len(set(SELECTED_PROPOSAL_ROWS)) != len(SELECTED_PROPOSAL_ROWS):
    raise ValueError("SELECTED_PROPOSAL_ROWS must not contain duplicates.")
if min(SELECTED_PROPOSAL_ROWS) < 1:
    raise ValueError("SELECTED_PROPOSAL_ROWS must use 1-based row numbers (minimum is 1).")

selected_rows_sorted = sorted(SELECTED_PROPOSAL_ROWS)
selected_row_set = set(selected_rows_sorted)

_BAD_PHRASES = [
    "Instruction:",
    "Input:",
    "Solution:",
    "```",
    "Please continue from where you left off.",
    "Continue the tutorial",
]
_BAD_IDS = [tokenizer(x, add_special_tokens=False).input_ids for x in _BAD_PHRASES]
_BAD_IDS = [x for x in _BAD_IDS if x]


def _infer_title_from_proposal(proposal: str) -> str:
    words = re.findall(r"[A-Za-z0-9]+", proposal)
    if not words:
        return "Project Overview"
    return " ".join(words[:4]).title() + " Project"


def _normalize_headings(text: str) -> str:
    normalized = text
    replacements = [
        (r"(?im)^\s*===\s*Title\s*===\s*$", "Title:"),
        (r"(?im)^\s*===\s*Summary\s*===\s*$", "Summary:"),
        (r"(?im)^\s*===\s*Roles\s*===\s*$", "Roles:"),
        (r"(?im)^\s*===\s*Features\s*===\s*$", "Features:"),
        (r"(?im)^\s*===\s*Goals\s*===\s*$", "Goals:"),
        (r"(?im)^\s*===\s*Timeline\s*===\s*$", "Timeline:"),
        (r"(?im)^\s*Project\s+Description\s*:\s*$", "Summary:"),
        (r"(?im)^\s*Timeline\s*\(Week\s*1-4\)\s*:\s*$", "Timeline:"),
        (r"(?im)^\s*Timeline\s*\(Week\s*2-4\)\s*:\s*$", "Timeline:"),
    ]
    for pattern, repl in replacements:
        normalized = re.sub(pattern, repl, normalized)

    if not re.search(r"(?im)^\s*Title\s*:", normalized):
        m = re.search(r"(?im)^\s*===\s*(.+?)\s*===\s*$", normalized)
        if m:
            title_text = m.group(1).strip()
            normalized = re.sub(r"(?im)^\s*===\s*(.+?)\s*===\s*$", f"Title: {title_text}", normalized, count=1)

    normalized = re.sub(r"(?im)^\s*Title\s*:\s*Title\s*:\s*", "Title: ", normalized)
    return normalized


def _dedupe_and_truncate_sections(text: str) -> tuple[str, list[str]]:
    heading_pattern = re.compile(r"^\s*(Title|Summary|Roles|Features|Goals|Timeline)\s*:", re.IGNORECASE)
    lines = text.splitlines()

    seen_headers = set()
    duplicate_headers = []
    cleaned_lines = []
    skip_duplicate_block = False
    timeline_seen = False

    for line in lines:
        m = heading_pattern.match(line)
        if m:
            header = m.group(1).lower()

            if timeline_seen and header != "timeline":
                break

            if header in seen_headers:
                duplicate_headers.append(header)
                skip_duplicate_block = True
                continue

            seen_headers.add(header)
            skip_duplicate_block = False
            if header == "timeline":
                timeline_seen = True
            cleaned_lines.append(line)
            continue

        if skip_duplicate_block:
            continue
        cleaned_lines.append(line)

    timeline_start = next(
        (i for i, line in enumerate(cleaned_lines) if re.match(r"^\s*Timeline\s*:", line, re.IGNORECASE)),
        None,
    )
    if timeline_start is not None:
        week4_idx = None
        for i in range(timeline_start, len(cleaned_lines)):
            if re.match(r"^\s*Week\s*4\s*:", cleaned_lines[i], re.IGNORECASE):
                week4_idx = i
                break
        if week4_idx is not None and week4_idx + 1 < len(cleaned_lines):
            cleaned_lines = cleaned_lines[:week4_idx + 1]

    cleaned_text = "\n".join(cleaned_lines).strip()
    cleaned_text = re.sub(r"(?im)^\s*Title\s*:\s*Title\s*:\s*", "Title: ", cleaned_text)
    return cleaned_text, sorted(set(duplicate_headers))


def _sanitize_response(response: str, proposal: str) -> str:
    response = response.split(OUTPUT_END_MARKER, 1)[0]
    lines = response.splitlines()

    first_section_idx = None
    for i, line in enumerate(lines):
        if re.match(r"^\s*(===.+===|Title\s*:|Summary\s*:|Roles\s*:|Features\s*:|Goals\s*:|Timeline\s*:)", line, re.IGNORECASE):
            first_section_idx = i
            break
    if first_section_idx is not None and first_section_idx > 0:
        lines = lines[first_section_idx:]

    cleaned = []
    skip = False
    for line in lines:
        if re.match(r"^\s*(Status|Backlog|Proposal\s*/\s*Input)\s*:", line, re.IGNORECASE):
            skip = True
            continue
        if re.match(r"^\s*(===\s*(Status|Backlog|Proposal|Requirement)\s*===)", line, re.IGNORECASE):
            skip = True
            continue
        if re.match(r"^\s*(===.+===|Title\s*:|Summary\s*:|Roles\s*:|Features\s*:|Goals\s*:|Timeline\s*:)", line, re.IGNORECASE):
            skip = False
        if not skip:
            cleaned.append(line)

    text = "\n".join(cleaned).strip()
    text = _normalize_headings(text)
    text, _ = _dedupe_and_truncate_sections(text)

    if not re.search(r"(?im)^\s*Title\s*:", text):
        title_guess = _infer_title_from_proposal(proposal)
        text = f"Title: {title_guess}\n" + text

    for sec in ["Summary", "Roles", "Features", "Goals", "Timeline"]:
        if not re.search(rf"(?im)^\s*{sec}\s*:", text):
            text += f"\n{sec}:\n"

    if re.search(r"(?im)^\s*Week\s*[1-4]\s*:", text) and not re.search(r"(?im)^\s*Timeline\s*:\s*$", text):
        text = re.sub(r"(?im)^\s*Week\s*1\s*:", "Timeline:\nWeek 1:", text, count=1)

    week_map = {}
    for wk, body in re.findall(r"(?im)^\s*Week\s*([1-4])\s*:\s*(.+)$", text):
        week_map[wk] = body.strip()

    if len(week_map) < 4:
        default_task = "Plan and execute key deliverables"
        timeline_block = ["Timeline:"]
        for w in ["1", "2", "3", "4"]:
            body = week_map.get(w, f"{default_task}, {default_task}")
            parts = [p.strip() for p in body.split(",") if p.strip()]
            if len(parts) == 1:
                parts.append(default_task)
            if len(parts) == 0:
                parts = [default_task, default_task]
            timeline_block.append(f"Week {w}: {parts[0]}, {parts[1]}")

        text = re.sub(
            r"(?ims)^\s*Timeline\s*:\s*[\s\S]*$",
            "\n".join(timeline_block),
            text,
            count=1,
        )

    text, _ = _dedupe_and_truncate_sections(text)
    return text.strip()


def _extract_sections(text: str) -> dict[str, str]:
    patterns = {
        "title": r"^Title\s*:\s*(.+)$",
        "summary": r"^Summary\s*:\s*([\s\S]*?)(?=^Roles\s*:|\Z)",
        "roles": r"^Roles\s*:\s*([\s\S]*?)(?=^Features\s*:|\Z)",
        "features": r"^Features\s*:\s*([\s\S]*?)(?=^Goals\s*:|\Z)",
        "goals": r"^Goals\s*:\s*([\s\S]*?)(?=^Timeline\s*:|\Z)",
        "timeline": r"^Timeline\s*:\s*([\s\S]*?)$",
    }
    out: dict[str, str] = {}

    for key, pat in patterns.items():
        m = re.search(pat, text, flags=re.MULTILINE | re.IGNORECASE)
        if m:
            out[key] = m.group(1).strip()

    week_lines = re.findall(r"(?im)^\s*Week\s*[1-4]\s*:\s*.+$", text)
    if ("timeline" not in out or not out.get("timeline")) and week_lines:
        out["timeline"] = "\n".join(week_lines)

    return out


def _has_duplicate_section_headers(response: str) -> bool:
    headers = re.findall(r"(?im)^\s*(Title|Summary|Roles|Features|Goals|Timeline)\s*:", response)
    counts = {}
    for header in headers:
        key = header.lower()
        counts[key] = counts.get(key, 0) + 1
    return any(v > 1 for v in counts.values())


def _is_generic_timeline(response: str) -> bool:
    """
    Check if timeline contains only generic filler phrases across weeks.
    Returns True if all weeks show generic content (suspicious), False otherwise.
    """
    week_matches = re.findall(r"(?im)^\s*Week\s*[1-4]\s*:\s*(.+)$", response)
    if not week_matches:
        return False  # No weeks found; can't claim it's generic

    if len(week_matches) < 4:
        return False  # Incomplete timeline

    generic_phrases = [
        "plan and execute",
        "plan execution",
        "execute tasks",
        "key deliverables",
        "implementation",
        "development",
    ]

    generic_count = 0
    for week_content in week_matches:
        content_lower = week_content.lower()
        # Check if content contains only generic phrases (no specifics)
        is_generic = any(phrase in content_lower for phrase in generic_phrases)
        if is_generic and len(week_content) < 50:  # Short + generic = suspicious
            generic_count += 1

    # If all 4 weeks look generic, flag it
    return generic_count == len(week_matches)


def _compliance_score(response: str) -> dict:
    """
    Score response on:
    1. All required sections present
    2. No forbidden sections (Status, Backlog, Proposal/Input)
    3. 4 unique weeks in timeline
    4. Timeline not all generic filler

    Returns dict with compliance check results.
    """
    sections = _extract_sections(response)
    required = ["title", "summary", "roles", "features", "goals", "timeline"]
    missing = [k for k in required if not sections.get(k)]

    # Check for forbidden sections in BOTH plain format (Status:) and heading format (=== Status ===)
    has_status = bool(
        re.search(r"(?im)^\s*(Status\s*:|===\s*Status\s*===)", response, flags=re.MULTILINE)
    )
    has_backlog = bool(
        re.search(r"(?im)^\s*(Backlog\s*:|===\s*Backlog\s*===)", response, flags=re.MULTILINE)
    )
    has_proposal = bool(
        re.search(r"(?im)^\s*(Proposal\s*/\s*Input\s*:|===\s*Proposal.*===|===\s*Requirements?\s*===)", response, flags=re.MULTILINE)
    )

    has_duplicate_sections = _has_duplicate_section_headers(response)

    # Check for generic timeline
    has_generic_timeline = _is_generic_timeline(response)

    week_matches = re.findall(r"^\s*Week\s*([1-4])\s*:\s*(.+)$", response, flags=re.MULTILINE | re.IGNORECASE)
    unique_weeks = {w for w, _ in week_matches}

    score = 0
    score += (6 - len(missing)) * 2
    score += len(unique_weeks)
    if not has_status:
        score += 1
    if not has_backlog:
        score += 1
    if not has_proposal:
        score += 1
    if not has_generic_timeline:
        score += 2  # Bonus for non-generic timeline

    return {
        "score": score,
        "missing": missing,
        "weeks": len(unique_weeks),
        "has_status": has_status,
        "has_backlog": has_backlog,
        "has_proposal": has_proposal,
        "has_generic_timeline": has_generic_timeline,
        "has_duplicate_sections": has_duplicate_sections,
    }


def _generate(prompt_text: str, max_new_tokens: int = 300) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model_infer.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.05,
            no_repeat_ngram_size=4,
            bad_words_ids=_BAD_IDS,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


dataset_file = DATASET_DIR / "model1_description_to_part1.jsonl"
examples_by_row = {}
max_row_seen = 0
with open(dataset_file, "r", encoding="utf-8") as f:
    for row_num, line in enumerate(f, start=1):
        max_row_seen = row_num
        if row_num not in selected_row_set:
            continue
        examples_by_row[row_num] = json.loads(line.strip())
        if len(examples_by_row) == len(selected_row_set):
            break

missing_rows = [row for row in selected_rows_sorted if row not in examples_by_row]
if missing_rows:
    raise ValueError(
        f"Selected row numbers out of range for dataset ({dataset_file}). "
        f"Requested rows: {selected_rows_sorted}, loaded through row {max_row_seen}, missing: {missing_rows}."
    )

examples = [examples_by_row[row] for row in selected_rows_sorted]

print("=" * 80)
print("MODEL 1 INFERENCE TEST - GROUNDED ON DATASET INPUTS")
print("=" * 80)
print(f"Quick mode: {QUICK_RUN_MODE}")
print(f"Strategies: {STRATEGIES_TO_RUN}")
print(f"Adapter path: {PROMOTED_ADAPTER_DIR}")
print(f"Selected dataset rows (1-based): {selected_rows_sorted}")
print(f"Examples to run: {len(examples)}")
print(f"Show full output: {SHOW_FULL_OUTPUT}")
print(f"Export txt: {EXPORT_OUTPUT_TXT}")
print(f"Few-shot examples loaded: {len(few_shot_examples)}")
print(f"Strict prompt integration active: {strict_prompt_integration_active}")
print(f"Strict prompt sections loaded: {strict_loaded_sections if strict_loaded_sections else 'None'}")
print(f"Strict prompt rule lines injected: {len(strict_rule_lines)}")
if USE_STRICT_PROMPT_RULES and not strict_prompt_integration_active:
    print("Strict prompt files were unavailable or yielded no compatible rules; using base guided prefix.")
print("=" * 80)

pass_count = 0
export_rows = []
for idx, example in enumerate(examples, start=1):
    proposal = example["prompt"]
    candidates = []

    if "A" in STRATEGIES_TO_RUN:
        raw_a = _generate(proposal, max_new_tokens=TOKENS_BY_STRATEGY["A"])
        clean_a = _sanitize_response(raw_a, proposal)
        score_a = _compliance_score(clean_a)
        candidates.append(("A: prompt-only", clean_a, score_a))

    if "B" in STRATEGIES_TO_RUN:
        prompt_b = FEWSHOT_CONTEXT + GUIDED_PREFIX + proposal
        raw_b = _generate(prompt_b, max_new_tokens=TOKENS_BY_STRATEGY["B"])
        clean_b = _sanitize_response(raw_b, proposal)
        score_b = _compliance_score(clean_b)
        candidates.append(("B: guided+few-shot", clean_b, score_b))

    if "C" in STRATEGIES_TO_RUN:
        prompt_c = FEWSHOT_CONTEXT + GUIDED_PREFIX + proposal + "\n\n==="
        raw_c = _generate(prompt_c, max_new_tokens=TOKENS_BY_STRATEGY["C"])
        clean_c = _sanitize_response("===" + raw_c, proposal)
        score_c = _compliance_score(clean_c)
        candidates.append(("C: guided+few-shot+anchor", clean_c, score_c))

    if not candidates:
        raise RuntimeError("No inference strategies selected. Set STRATEGIES_TO_RUN to at least one of A/B/C.")

    best_name, best_out, best_score = max(candidates, key=lambda x: x[2]["score"])

    compliant = (
        len(best_score["missing"]) == 0
        and best_score["weeks"] == 4
        and not best_score["has_status"]
        and not best_score["has_backlog"]
        and not best_score["has_proposal"]
        and not best_score["has_generic_timeline"]
    )

    print(f"\n{'=' * 80}")
    print(f"EXAMPLE {idx} (dataset row {selected_rows_sorted[idx - 1]})")
    print(f"Best strategy: {best_name} (score={best_score['score']})")
    print(
        f"Sections missing: {best_score['missing'] if best_score['missing'] else 'None'} | "
        f"Weeks: {best_score['weeks']}/4 | "
        f"Status: {best_score['has_status']} | "
        f"Backlog: {best_score['has_backlog']} | "
        f"Proposal: {best_score['has_proposal']} | "
        f"Generic timeline: {best_score['has_generic_timeline']} | "
        f"Duplicate sections: {best_score['has_duplicate_sections']}"
    )
    print(f"Result: {'PASS' if compliant else 'FAIL'}")
    print("Output:")
    if SHOW_FULL_OUTPUT:
        print(best_out)
    else:
        print(best_out[:320])

    export_rows.append({
        "example": idx,
        "dataset_row": selected_rows_sorted[idx - 1],
        "best_strategy": best_name,
        "score": best_score["score"],
        "compliant": compliant,
        "missing": best_score["missing"],
        "weeks": best_score["weeks"],
        "has_status": best_score["has_status"],
        "has_backlog": best_score["has_backlog"],
        "has_proposal": best_score["has_proposal"],
        "has_generic_timeline": best_score["has_generic_timeline"],
        "has_duplicate_sections": best_score["has_duplicate_sections"],
        "proposal": proposal,
        "output": best_out,
    })

    if compliant:
        pass_count += 1

print(f"\n{'=' * 80}")
print(f"COMPLIANCE SUMMARY: {pass_count}/{len(examples)} fully compliant")
print("=" * 80)

if EXPORT_OUTPUT_TXT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    export_file = EXPORT_DIR / f"model1_inference_outputs_{stamp}.txt"

    lines = []
    lines.append("MODEL 1 INFERENCE TEST EXPORT")
    lines.append(f"Generated: {datetime.now().isoformat(timespec='seconds')}")
    lines.append(f"Dataset: {dataset_file}")
    lines.append(f"Selected rows (1-based): {selected_rows_sorted}")
    lines.append(f"Strict prompt integration active: {strict_prompt_integration_active}")
    lines.append(f"Strict prompt sections loaded: {strict_loaded_sections if strict_loaded_sections else 'None'}")
    lines.append(f"Strict prompt rule lines injected: {len(strict_rule_lines)}")
    lines.append(f"Examples run: {len(examples)}")
    lines.append(f"Compliance summary: {pass_count}/{len(examples)}")
    lines.append("=" * 100)

    for row in export_rows:
        lines.append("")
        lines.append("-" * 100)
        lines.append(f"EXAMPLE {row['example']} (dataset row {row['dataset_row']})")
        lines.append(f"Best strategy: {row['best_strategy']} | Score: {row['score']} | Compliant: {row['compliant']}")
        lines.append(
            "Checks: "
            f"missing={row['missing'] if row['missing'] else 'None'}, "
            f"weeks={row['weeks']}/4, "
            f"status={row['has_status']}, backlog={row['has_backlog']}, proposal={row['has_proposal']}, "
            f"generic_timeline={row['has_generic_timeline']}, duplicate_sections={row['has_duplicate_sections']}"
        )
        lines.append("")
        lines.append("Proposal / Input:")
        lines.append(row["proposal"].strip())
        lines.append("")
        lines.append("Model Output:")
        lines.append(row["output"].strip())

    export_file.write_text("\n".join(lines), encoding="utf-8")
    print(f"\nSaved inference outputs to: {export_file}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MODEL 1 INFERENCE TEST - GROUNDED ON DATASET INPUTS
Quick mode: False
Strategies: ['A', 'B', 'C']
Adapter path: C:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model1_overview_lora_1p5b
Selected dataset rows (1-based): [6, 7, 8, 9, 10]
Examples to run: 5
Show full output: True
Export txt: True
Few-shot examples loaded: 2
Strict prompt integration active: True
Strict prompt sections loaded: ['summary', 'roles', 'features', 'goals', 'timeline']
Strict prompt rule lines injected: 20

EXAMPLE 1 (dataset row 6)
Best strategy: B: guided+few-shot (score=21)
Sections missing: None | Weeks: 4/4 | Status: False | Backlog: False | Proposal: False | Generic timeline: False | Duplicate sections: False
Result: PASS
Output:
Title: SafeSchool Campus Safety System
Summary:
SafeSchool is a campus safety system designed to monitor entrances, detect activities, and send alerts in case of emergencies. It aims to improve response times, reduce incidents, and ensure a safer environment.
Ro

## Step 7: Quick inference test (compact strict prompts)

Copy of Step 6 with lower prompt noise and strict formatting preserved. This variant removes few-shot context, keeps strict prompt integration active, and uses a tighter strict-rule cap.

In [9]:
import gc
import json
import os
import re
from datetime import datetime

# Final VRAM cleanup before loading inference models.
for _name in ("eval_trainer", "winner_model", "eval_base", "model", "trainer"):
    if _name in globals():
        del globals()[_name]
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

adapter_rel = os.getenv("PEFT_ADAPTER_PATH", "llms/fine_tune/qwen_model1_overview_lora_1p5b")
adapter_path_obj = Path(adapter_rel)
if adapter_path_obj.is_absolute():
    PROMOTED_ADAPTER_DIR = adapter_path_obj
else:
    PROMOTED_ADAPTER_DIR = (_ai_root / adapter_path_obj).resolve()

if not PROMOTED_ADAPTER_DIR.exists():
    raise FileNotFoundError(f"Promoted adapter not found: {PROMOTED_ADAPTER_DIR}")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
model_infer = PeftModel.from_pretrained(base_model, str(PROMOTED_ADAPTER_DIR))
if torch.cuda.is_available():
    model_infer = model_infer.to("cuda")

model_infer.eval()
for param in model_infer.parameters():
    param.requires_grad = False

# Step 7 compact mode intentionally removes few-shot text to reduce prompt noise.
few_shot_examples = []
FEWSHOT_CONTEXT = ""

PROMPTS_DIR = _ai_root / "llms" / "prompts"
STRICT_PROMPT_SECTIONS = ["summary", "roles", "features", "goals", "timeline"]
USE_STRICT_PROMPT_RULES = True
MAX_RULE_LINES_PER_SECTION = 4
OUTPUT_END_MARKER = "END_OF_OVERVIEW"
EXTRA_STRICT_RULE_LINES = [
    "Use each required heading exactly once; do not repeat sections.",
    "Goals must contain exactly 5 lines and each line must start with a strong verb.",
    "Timeline must include Week 1 to Week 4; each week has exactly two distinct tasks tied to Goals or Features.",
    "Do not reuse the same week text; avoid repeating task phrases across weeks.",
    "After Week 4, output END_OF_OVERVIEW and stop.",
]

VERB_HINTS = {
    "build", "define", "design", "implement", "create", "develop", "integrate", "test",
    "deploy", "optimize", "monitor", "configure", "validate", "automate", "improve", "launch",
    "deliver", "establish", "set", "enable", "analyze", "document"
}

DEFAULT_ROLES = ["Project Manager", "Backend Developer", "Frontend Developer"]


def _load_prompt_template(section):
    prompt_path = PROMPTS_DIR / f"{section}_prompt.txt"
    if not prompt_path.exists():
        return None
    try:
        return prompt_path.read_text(encoding="utf-8")
    except OSError:
        return None


def _extract_section_rules(section, template):
    _ = section
    lines = template.splitlines()
    rules = []
    in_rules_block = False

    for line in lines:
        stripped = line.strip()
        lowered = stripped.lower()

        if not stripped:
            continue
        if stripped.startswith("<<<"):
            continue

        if lowered.startswith("rules:"):
            in_rules_block = True
            continue

        if lowered.startswith("example format:"):
            in_rules_block = False
            continue

        if in_rules_block:
            m = re.match(r"^\d+\.\s*(.+)$", stripped)
            if m:
                rules.append(m.group(1).strip())

    return rules


def _normalize_rule_for_combined_output(section, rule):
    lowered = rule.lower()

    skip_markers = [
        "exactly two spaces",
        "exactly four spaces",
        "output only the yaml block",
        "example format",
        "<<<",
        "{proposal_text}",
        "{roles}",
        "{additional_roles}",
    ]
    if any(marker in lowered for marker in skip_markers):
        return None

    if section == "goals":
        if "epic" in lowered or "goal" in lowered:
            return "Goals must be project objectives and not duplicates of Features."

    if section == "timeline":
        if "exactly 4 weeks" in lowered or "week" in lowered:
            return "Timeline must include Week 1, Week 2, Week 3, and Week 4 exactly once."
        if "exactly 2 tasks" in lowered:
            return "Each week must include exactly two distinct, concrete tasks."

    if section == "summary":
        if "2-3" in lowered and "sentence" in lowered:
            return "Keep Summary concise at about 2-3 sentences."

    if "title case" in lowered and section == "roles":
        return "Use Title Case role names in Roles."

    return None


def _build_strict_constraints_block():
    loaded_sections = []
    all_rules = []

    for section in STRICT_PROMPT_SECTIONS:
        template = _load_prompt_template(section)
        if not template:
            continue

        loaded_sections.append(section)
        raw_rules = _extract_section_rules(section, template)

        normalized = []
        for rule in raw_rules:
            mapped = _normalize_rule_for_combined_output(section, rule)
            if mapped and mapped not in normalized:
                normalized.append(mapped)

        all_rules.extend(normalized[:MAX_RULE_LINES_PER_SECTION])

    deduped = []
    seen = set()
    for rule in all_rules + EXTRA_STRICT_RULE_LINES:
        if rule not in seen:
            deduped.append(rule)
            seen.add(rule)

    if not deduped:
        return "", loaded_sections, []

    block = "Additional strict format guidance:\n"
    block += "\n".join(f"- {rule}" for rule in deduped)
    block += "\n\n"
    return block, loaded_sections, deduped


BASE_GUIDED_PREFIX = (
    "Generate ONLY the following sections in order: "
    "Title, Summary, Roles, Features, Goals, Timeline (Week 1-4).\n"
    "Use each heading once only; do not repeat sections.\n"
    "Do not include Status, Proposal/Input, or Backlog.\n"
    "Goals section rules: exactly 5 bullet lines, each line starts with a verb.\n"
    "Timeline rules: Week 1 to Week 4 only; each week has exactly two distinct tasks tied to Goals or Features.\n"
    "Do not repeat the same week text or same task phrase across multiple weeks.\n"
    f"After Week 4, output only '{OUTPUT_END_MARKER}' on its own line and stop.\n\n"
)

strict_constraints_block, strict_loaded_sections, strict_rule_lines = _build_strict_constraints_block()
strict_prompt_integration_active = bool(USE_STRICT_PROMPT_RULES and strict_rule_lines)

if strict_prompt_integration_active:
    GUIDED_PREFIX = BASE_GUIDED_PREFIX + strict_constraints_block
else:
    GUIDED_PREFIX = BASE_GUIDED_PREFIX

# Inference controls
QUICK_RUN_MODE = True  # If True, runs only strategy B with fewer tokens for faster iteration during development.
ENABLE_CONDITIONAL_C_FALLBACK = True
STRATEGIES_TO_RUN = ["B"] if QUICK_RUN_MODE else ["A", "B", "C"]
TOKENS_BY_STRATEGY = {"A": 260, "B": 350, "C": 400}
SHOW_FULL_OUTPUT = True
EXPORT_OUTPUT_TXT = True
EXPORT_DIR = PROMOTED_ADAPTER_DIR / "inference_test_exports_step7_compact_strict"

# Deterministic proposal selection (1-based dataset row numbers).
SELECTED_PROPOSAL_ROWS = [6, 7, 8, 9, 10]

if len(SELECTED_PROPOSAL_ROWS) != 5:
    raise ValueError(
        "SELECTED_PROPOSAL_ROWS must contain exactly 5 row numbers "
        f"(received {len(SELECTED_PROPOSAL_ROWS)})."
    )
if not all(isinstance(x, int) for x in SELECTED_PROPOSAL_ROWS):
    raise ValueError("SELECTED_PROPOSAL_ROWS must contain integers only.")
if len(set(SELECTED_PROPOSAL_ROWS)) != len(SELECTED_PROPOSAL_ROWS):
    raise ValueError("SELECTED_PROPOSAL_ROWS must not contain duplicates.")
if min(SELECTED_PROPOSAL_ROWS) < 1:
    raise ValueError("SELECTED_PROPOSAL_ROWS must use 1-based row numbers (minimum is 1).")

selected_rows_sorted = sorted(SELECTED_PROPOSAL_ROWS)
selected_row_set = set(selected_rows_sorted)

_BAD_PHRASES = [
    "Instruction:",
    "Input:",
    "Solution:",
    "```",
    "Please continue from where you left off.",
    "Continue the tutorial",
]
_BAD_IDS = [tokenizer(x, add_special_tokens=False).input_ids for x in _BAD_PHRASES]
_BAD_IDS = [x for x in _BAD_IDS if x]


def _infer_title_from_proposal(proposal: str) -> str:
    words = re.findall(r"[A-Za-z0-9]+", proposal)
    if not words:
        return "Project Overview"
    return " ".join(words[:4]).title() + " Project"


def _normalize_headings(text: str) -> str:
    normalized = text
    replacements = [
        (r"(?im)^\s*===\s*Title\s*===\s*$", "Title:"),
        (r"(?im)^\s*===\s*Summary\s*===\s*$", "Summary:"),
        (r"(?im)^\s*===\s*Roles\s*===\s*$", "Roles:"),
        (r"(?im)^\s*===\s*Features\s*===\s*$", "Features:"),
        (r"(?im)^\s*===\s*Goals\s*===\s*$", "Goals:"),
        (r"(?im)^\s*===\s*Timeline\s*===\s*$", "Timeline:"),
        (r"(?im)^\s*Project\s+Description\s*:\s*$", "Summary:"),
        (r"(?im)^\s*Timeline\s*\(Week\s*1-4\)\s*:\s*$", "Timeline:"),
        (r"(?im)^\s*Timeline\s*\(Week\s*2-4\)\s*:\s*$", "Timeline:"),
    ]
    for pattern, repl in replacements:
        normalized = re.sub(pattern, repl, normalized)

    if not re.search(r"(?im)^\s*Title\s*:", normalized):
        m = re.search(r"(?im)^\s*===\s*(.+?)\s*===\s*$", normalized)
        if m:
            title_text = m.group(1).strip()
            normalized = re.sub(r"(?im)^\s*===\s*(.+?)\s*===\s*$", f"Title: {title_text}", normalized, count=1)

    normalized = re.sub(r"(?im)^\s*Title\s*:\s*Title\s*:\s*", "Title: ", normalized)
    return normalized


def _dedupe_and_truncate_sections(text: str) -> tuple[str, list[str]]:
    heading_pattern = re.compile(r"^\s*(Title|Summary|Roles|Features|Goals|Timeline)\s*:", re.IGNORECASE)
    lines = text.splitlines()

    seen_headers = set()
    duplicate_headers = []
    cleaned_lines = []
    skip_duplicate_block = False
    timeline_seen = False

    for line in lines:
        m = heading_pattern.match(line)
        if m:
            header = m.group(1).lower()

            if timeline_seen and header != "timeline":
                break

            if header in seen_headers:
                duplicate_headers.append(header)
                skip_duplicate_block = True
                continue

            seen_headers.add(header)
            skip_duplicate_block = False
            if header == "timeline":
                timeline_seen = True
            cleaned_lines.append(line)
            continue

        if skip_duplicate_block:
            continue
        cleaned_lines.append(line)

    timeline_start = next(
        (i for i, line in enumerate(cleaned_lines) if re.match(r"^\s*Timeline\s*:", line, re.IGNORECASE)),
        None,
    )
    if timeline_start is not None:
        week4_idx = None
        for i in range(timeline_start, len(cleaned_lines)):
            if re.match(r"^\s*Week\s*4\s*:", cleaned_lines[i], re.IGNORECASE):
                week4_idx = i
                break
        if week4_idx is not None and week4_idx + 1 < len(cleaned_lines):
            cleaned_lines = cleaned_lines[:week4_idx + 1]

    cleaned_text = "\n".join(cleaned_lines).strip()
    cleaned_text = re.sub(r"(?im)^\s*Title\s*:\s*Title\s*:\s*", "Title: ", cleaned_text)
    return cleaned_text, sorted(set(duplicate_headers))


def _default_timeline_block() -> list[str]:
    return [
        "Timeline:",
        "Week 1: Define scope from goals, Design feature architecture",
        "Week 2: Implement core backend flows, Build frontend interfaces",
        "Week 3: Integrate modules and APIs, Validate end-to-end behavior",
        "Week 4: Run QA and bug fixes, Deploy and monitor release",
    ]


def _extract_role_items(roles_block: str) -> list[str]:
    """Extract individual role names from roles section text."""
    if not roles_block.strip():
        return []
    roles = []
    for line in roles_block.splitlines():
        text = line.strip()
        if not text:
            continue
        text = re.sub(r"^[-*\d\.\)\s]+", "", text).strip()
        if text:
            roles.append(text)
    return roles


def _role_category_hits(roles_list: list[str]) -> dict[str, bool]:
    """Map roles into required categories using keyword matching."""
    roles_lower = [r.lower() for r in roles_list]

    return {
        "project": any("project" in role for role in roles_lower),
        "backend": any("backend" in role for role in roles_lower),
        "frontend": any(("frontend" in role) or ("front end" in role) for role in roles_lower),
    }


def _has_all_default_roles(roles_list: list[str]) -> tuple[bool, int]:
    """
    Check if all default role categories are present and count them.
    Categories: project, backend, frontend/front end.
    Returns: (has_all_3, count_of_defaults)
    """
    hits = _role_category_hits(roles_list)
    found_count = sum(1 for v in hits.values() if v)
    return found_count == 3, found_count


def _backfill_roles(response: str) -> str:
    """Ensure all default roles are present by backfilling missing ones."""
    sections = _extract_sections(response)
    roles_block = sections.get("roles", "")
    role_items = _extract_role_items(roles_block)
    
    has_all, count = _has_all_default_roles(role_items)
    if has_all:
        return response
    
    # Backfill only missing role categories with canonical defaults
    hits = _role_category_hits(role_items)
    category_defaults = [
        ("project", "Project Manager"),
        ("backend", "Backend Developer"),
        ("frontend", "Frontend Developer"),
    ]
    missing_defaults = [default for category, default in category_defaults if not hits[category]]
    
    if missing_defaults:
        # Find Roles section in response and append missing defaults
        roles_pattern = r"(^Roles\s*:\s*)(.*?)(?=^[A-Z][a-zA-Z]+\s*:|$)"
        def add_missing(match):
            header = match.group(1)
            content = match.group(2)
            for role in missing_defaults:
                if not re.search(rf"(?i){re.escape(role)}", content):
                    content = content.rstrip() + f"\n- {role}"
            return header + content
        
        response = re.sub(roles_pattern, add_missing, response, flags=re.MULTILINE | re.IGNORECASE)
    
    return response


def _sanitize_response(response: str, proposal: str) -> str:
    response = response.split(OUTPUT_END_MARKER, 1)[0]
    lines = response.splitlines()

    first_section_idx = None
    for i, line in enumerate(lines):
        if re.match(r"^\s*(===.+===|Title\s*:|Summary\s*:|Roles\s*:|Features\s*:|Goals\s*:|Timeline\s*:)", line, re.IGNORECASE):
            first_section_idx = i
            break
    if first_section_idx is not None and first_section_idx > 0:
        lines = lines[first_section_idx:]

    cleaned = []
    skip = False
    for line in lines:
        if re.match(r"^\s*(Status|Backlog|Proposal\s*/\s*Input)\s*:", line, re.IGNORECASE):
            skip = True
            continue
        if re.match(r"^\s*(===\s*(Status|Backlog|Proposal|Requirement)\s*===)", line, re.IGNORECASE):
            skip = True
            continue
        if re.match(r"^\s*(===.+===|Title\s*:|Summary\s*:|Roles\s*:|Features\s*:|Goals\s*:|Timeline\s*:)", line, re.IGNORECASE):
            skip = False
        if not skip:
            cleaned.append(line)

    text = "\n".join(cleaned).strip()
    text = _normalize_headings(text)
    text, _ = _dedupe_and_truncate_sections(text)

    if not re.search(r"(?im)^\s*Title\s*:", text):
        title_guess = _infer_title_from_proposal(proposal)
        text = f"Title: {title_guess}\n" + text

    for sec in ["Summary", "Roles", "Features", "Goals", "Timeline"]:
        if not re.search(rf"(?im)^\s*{sec}\s*:", text):
            text += f"\n{sec}:\n"

    if re.search(r"(?im)^\s*Week\s*[1-4]\s*:", text) and not re.search(r"(?im)^\s*Timeline\s*:\s*$", text):
        text = re.sub(r"(?im)^\s*Week\s*1\s*:", "Timeline:\nWeek 1:", text, count=1)

    week_map = {}
    for wk, body in re.findall(r"(?im)^\s*Week\s*([1-4])\s*:\s*(.+)$", text):
        week_map[wk] = body.strip()

    if len(week_map) < 4:
        text = re.sub(
            r"(?ims)^\s*Timeline\s*:\s*[\s\S]*$",
            "\n".join(_default_timeline_block()),
            text,
            count=1,
        )

    text, _ = _dedupe_and_truncate_sections(text)
    text = text.strip()
    
    # Backfill missing default roles if Roles section exists
    text = _backfill_roles(text)
    
    return text


def _extract_sections(text: str) -> dict[str, str]:
    patterns = {
        "title": r"^Title\s*:\s*(.+)$",
        "summary": r"^Summary\s*:\s*([\s\S]*?)(?=^Roles\s*:|\Z)",
        "roles": r"^Roles\s*:\s*([\s\S]*?)(?=^Features\s*:|\Z)",
        "features": r"^Features\s*:\s*([\s\S]*?)(?=^Goals\s*:|\Z)",
        "goals": r"^Goals\s*:\s*([\s\S]*?)(?=^Timeline\s*:|\Z)",
        "timeline": r"^Timeline\s*:\s*([\s\S]*?)$",
    }
    out: dict[str, str] = {}

    for key, pat in patterns.items():
        m = re.search(pat, text, flags=re.MULTILINE | re.IGNORECASE)
        if m:
            out[key] = m.group(1).strip()

    week_lines = re.findall(r"(?im)^\s*Week\s*[1-4]\s*:\s*.+$", text)
    if ("timeline" not in out or not out.get("timeline")) and week_lines:
        out["timeline"] = "\n".join(week_lines)

    return out


def _extract_goal_lines(goals_block: str) -> list[str]:
    if not goals_block.strip():
        return []
    lines = []
    for raw in goals_block.splitlines():
        line = raw.strip()
        if not line:
            continue
        line = re.sub(r"^[-*\d\.\)\s]+", "", line).strip()
        if line:
            lines.append(line)
    return lines


def _is_verb_first(line: str) -> bool:
    m = re.match(r"^([A-Za-z]+)", line.strip())
    if not m:
        return False
    first = m.group(1).lower()
    return first in VERB_HINTS or first.endswith("ing")


def _has_duplicate_section_headers(response: str) -> bool:
    headers = re.findall(r"(?im)^\s*(Title|Summary|Roles|Features|Goals|Timeline)\s*:", response)
    counts = {}
    for header in headers:
        key = header.lower()
        counts[key] = counts.get(key, 0) + 1
    return any(v > 1 for v in counts.values())


def _week_bodies(response: str) -> list[str]:
    week_matches = re.findall(r"(?im)^\s*Week\s*([1-4])\s*:\s*(.+)$", response)
    week_map = {}
    for week, body in week_matches:
        week_map[week] = re.sub(r"\s+", " ", body.strip().lower())
    return [week_map[w] for w in ["1", "2", "3", "4"] if w in week_map]


def _has_repeated_week_text(response: str) -> bool:
    bodies = _week_bodies(response)
    if len(bodies) < 4:
        return False
    return len(set(bodies)) < len(bodies)


def _is_generic_timeline(response: str) -> bool:
    bodies = _week_bodies(response)
    if len(bodies) < 4:
        return False

    generic_phrases = [
        "plan and execute",
        "plan execution",
        "execute tasks",
        "key deliverables",
        "implementation",
        "development",
    ]

    generic_hits = 0
    for body in bodies:
        if any(phrase in body for phrase in generic_phrases):
            generic_hits += 1

    return generic_hits >= 2 or _has_repeated_week_text(response)


def _compliance_score(response: str) -> dict:
    sections = _extract_sections(response)
    required = ["title", "summary", "roles", "features", "goals", "timeline"]
    missing = [k for k in required if not sections.get(k)]

    has_status = bool(
        re.search(r"(?im)^\s*(Status\s*:|===\s*Status\s*===)", response, flags=re.MULTILINE)
    )
    has_backlog = bool(
        re.search(r"(?im)^\s*(Backlog\s*:|===\s*Backlog\s*===)", response, flags=re.MULTILINE)
    )
    has_proposal = bool(
        re.search(r"(?im)^\s*(Proposal\s*/\s*Input\s*:|===\s*Proposal.*===|===\s*Requirements?\s*===)", response, flags=re.MULTILINE)
    )

    has_duplicate_sections = _has_duplicate_section_headers(response)
    has_generic_timeline = _is_generic_timeline(response)
    has_repeated_weeks = _has_repeated_week_text(response)

    goal_lines = _extract_goal_lines(sections.get("goals", ""))
    goals_count_issue = len(goal_lines) != 5
    goals_verb_issue = any(not _is_verb_first(line) for line in goal_lines) if goal_lines else True
    
    role_items = _extract_role_items(sections.get("roles", ""))
    has_all_defaults, default_roles_count = _has_all_default_roles(role_items)

    week_matches = re.findall(r"^\s*Week\s*([1-4])\s*:\s*(.+)$", response, flags=re.MULTILINE | re.IGNORECASE)
    unique_weeks = {w for w, _ in week_matches}

    score = 0
    score += (6 - len(missing)) * 2
    score += len(unique_weeks)
    if not has_status:
        score += 1
    if not has_backlog:
        score += 1
    if not has_proposal:
        score += 1
    if not has_generic_timeline:
        score += 2
    if not goals_count_issue:
        score += 2
    if not goals_verb_issue:
        score += 1
    if has_all_defaults:
        score += 2
    elif default_roles_count == 2:
        score += 1

    return {
        "score": score,
        "missing": missing,
        "weeks": len(unique_weeks),
        "has_status": has_status,
        "has_backlog": has_backlog,
        "has_proposal": has_proposal,
        "has_generic_timeline": has_generic_timeline,
        "has_repeated_weeks": has_repeated_weeks,
        "has_duplicate_sections": has_duplicate_sections,
        "goals_count_issue": goals_count_issue,
        "goals_verb_issue": goals_verb_issue,
        "goal_lines": goal_lines,
        "has_default_roles": has_all_defaults,
        "default_roles_count": default_roles_count,
    }


def _fails_quality_gate(score: dict) -> bool:
    return (
        len(score["missing"]) > 0
        or score["weeks"] != 4
        or score["has_status"]
        or score["has_backlog"]
        or score["has_proposal"]
        or score["has_generic_timeline"]
        or score["has_repeated_weeks"]
        or score["goals_count_issue"]
        or score["goals_verb_issue"]
    )


def _generate(prompt_text: str, max_new_tokens: int = 300) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model_infer.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.05,
            no_repeat_ngram_size=4,
            bad_words_ids=_BAD_IDS,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


dataset_file = DATASET_DIR / "model1_description_to_part1.jsonl"
examples_by_row = {}
max_row_seen = 0
with open(dataset_file, "r", encoding="utf-8") as f:
    for row_num, line in enumerate(f, start=1):
        max_row_seen = row_num
        if row_num not in selected_row_set:
            continue
        examples_by_row[row_num] = json.loads(line.strip())
        if len(examples_by_row) == len(selected_row_set):
            break

missing_rows = [row for row in selected_rows_sorted if row not in examples_by_row]
if missing_rows:
    raise ValueError(
        f"Selected row numbers out of range for dataset ({dataset_file}). "
        f"Requested rows: {selected_rows_sorted}, loaded through row {max_row_seen}, missing: {missing_rows}."
    )

examples = [examples_by_row[row] for row in selected_rows_sorted]

print("=" * 80)
print("MODEL 1 INFERENCE TEST - STEP 7 COMPACT STRICT PROMPTS")
print("=" * 80)
print(f"Quick mode: {QUICK_RUN_MODE}")
print(f"Base strategies: {STRATEGIES_TO_RUN}")
print(f"Conditional C fallback: {ENABLE_CONDITIONAL_C_FALLBACK}")
print(f"Adapter path: {PROMOTED_ADAPTER_DIR}")
print(f"Selected dataset rows (1-based): {selected_rows_sorted}")
print(f"Examples to run: {len(examples)}")
print(f"Show full output: {SHOW_FULL_OUTPUT}")
print(f"Export txt: {EXPORT_OUTPUT_TXT}")
print("Few-shot examples loaded: 0 (disabled for compact strict prompts)")
print(f"Strict prompt integration active: {strict_prompt_integration_active}")
print(f"Strict prompt sections loaded: {strict_loaded_sections if strict_loaded_sections else 'None'}")
print(f"Strict prompt rule lines injected: {len(strict_rule_lines)}")
if USE_STRICT_PROMPT_RULES and not strict_prompt_integration_active:
    print("Strict prompt files were unavailable or yielded no compatible rules; using base guided prefix.")
print("=" * 80)

pass_count = 0
export_rows = []
for idx, example in enumerate(examples, start=1):
    proposal = example["prompt"]
    candidates = []

    if "A" in STRATEGIES_TO_RUN:
        raw_a = _generate(proposal, max_new_tokens=TOKENS_BY_STRATEGY["A"])
        clean_a = _sanitize_response(raw_a, proposal)
        score_a = _compliance_score(clean_a)
        candidates.append(("A: prompt-only", clean_a, score_a))

    b_score = None
    if "B" in STRATEGIES_TO_RUN:
        prompt_b = GUIDED_PREFIX + proposal
        raw_b = _generate(prompt_b, max_new_tokens=TOKENS_BY_STRATEGY["B"])
        clean_b = _sanitize_response(raw_b, proposal)
        b_score = _compliance_score(clean_b)
        candidates.append(("B: guided+compact-strict", clean_b, b_score))

    should_run_c = "C" in STRATEGIES_TO_RUN
    if (
        not should_run_c
        and ENABLE_CONDITIONAL_C_FALLBACK
        and b_score is not None
        and _fails_quality_gate(b_score)
    ):
        should_run_c = True

    if should_run_c:
        prompt_c = GUIDED_PREFIX + proposal + "\n\n==="
        raw_c = _generate(prompt_c, max_new_tokens=TOKENS_BY_STRATEGY["C"])
        clean_c = _sanitize_response("===" + raw_c, proposal)
        score_c = _compliance_score(clean_c)
        label_c = "C: conditional-fallback-anchor" if "C" not in STRATEGIES_TO_RUN else "C: guided+compact-strict+anchor"
        candidates.append((label_c, clean_c, score_c))

    if not candidates:
        raise RuntimeError("No inference strategies selected. Set STRATEGIES_TO_RUN to include B or others.")

    best_name, best_out, best_score = max(candidates, key=lambda x: x[2]["score"])

    compliant = (
        len(best_score["missing"]) == 0
        and best_score["weeks"] == 4
        and not best_score["has_status"]
        and not best_score["has_backlog"]
        and not best_score["has_proposal"]
        and not best_score["has_generic_timeline"]
        and not best_score["has_repeated_weeks"]
        and not best_score["goals_count_issue"]
        and not best_score["goals_verb_issue"]
    )

    print(f"\n{'=' * 80}")
    print(f"EXAMPLE {idx} (dataset row {selected_rows_sorted[idx - 1]})")
    print(f"Best strategy: {best_name} (score={best_score['score']})")
    print(
        f"Sections missing: {best_score['missing'] if best_score['missing'] else 'None'} | "
        f"Weeks: {best_score['weeks']}/4 | "
        f"Status: {best_score['has_status']} | "
        f"Backlog: {best_score['has_backlog']} | "
        f"Proposal: {best_score['has_proposal']} | "
        f"Generic timeline: {best_score['has_generic_timeline']} | "
        f"Repeated weeks: {best_score['has_repeated_weeks']} | "
        f"Goals count issue: {best_score['goals_count_issue']} | "
        f"Goals verb issue: {best_score['goals_verb_issue']} | "
        f"Has default roles: {best_score['has_default_roles']} | "
        f"Default roles count: {best_score['default_roles_count']} | "
        f"Duplicate sections: {best_score['has_duplicate_sections']}"
    )
    print(f"Result: {'PASS' if compliant else 'FAIL'}")
    print("Output:")
    if SHOW_FULL_OUTPUT:
        print(best_out)
    else:
        print(best_out[:320])

    export_rows.append({
        "example": idx,
        "dataset_row": selected_rows_sorted[idx - 1],
        "best_strategy": best_name,
        "score": best_score["score"],
        "compliant": compliant,
        "missing": best_score["missing"],
        "weeks": best_score["weeks"],
        "has_status": best_score["has_status"],
        "has_backlog": best_score["has_backlog"],
        "has_proposal": best_score["has_proposal"],
        "has_generic_timeline": best_score["has_generic_timeline"],
        "has_repeated_weeks": best_score["has_repeated_weeks"],
        "goals_count_issue": best_score["goals_count_issue"],
        "goals_verb_issue": best_score["goals_verb_issue"],
        "has_duplicate_sections": best_score["has_duplicate_sections"],
        "has_default_roles": best_score["has_default_roles"],
        "default_roles_count": best_score["default_roles_count"],
        "proposal": proposal,
        "output": best_out,
    })

    if compliant:
        pass_count += 1

print(f"\n{'=' * 80}")
print(f"COMPLIANCE SUMMARY: {pass_count}/{len(examples)} fully compliant")
print("=" * 80)

if EXPORT_OUTPUT_TXT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    export_file = EXPORT_DIR / f"model1_inference_outputs_step7_compact_strict_{stamp}.txt"

    lines = []
    lines.append("MODEL 1 INFERENCE TEST EXPORT - STEP 7 COMPACT STRICT")
    lines.append(f"Generated: {datetime.now().isoformat(timespec='seconds')}")
    lines.append(f"Dataset: {dataset_file}")
    lines.append(f"Selected rows (1-based): {selected_rows_sorted}")
    lines.append(f"Strict prompt integration active: {strict_prompt_integration_active}")
    lines.append(f"Strict prompt sections loaded: {strict_loaded_sections if strict_loaded_sections else 'None'}")
    lines.append(f"Strict prompt rule lines injected: {len(strict_rule_lines)}")
    lines.append(f"Examples run: {len(examples)}")
    lines.append(f"Compliance summary: {pass_count}/{len(examples)}")
    lines.append("=" * 100)

    for row in export_rows:
        lines.append("")
        lines.append("-" * 100)
        lines.append(f"EXAMPLE {row['example']} (dataset row {row['dataset_row']})")
        lines.append(f"Best strategy: {row['best_strategy']} | Score: {row['score']} | Compliant: {row['compliant']}")
        lines.append(
            "Checks: "
            f"missing={row['missing'] if row['missing'] else 'None'}, "
            f"weeks={row['weeks']}/4, "
            f"status={row['has_status']}, backlog={row['has_backlog']}, proposal={row['has_proposal']}, "
            f"generic_timeline={row['has_generic_timeline']}, repeated_weeks={row['has_repeated_weeks']}, "
            f"goals_count_issue={row['goals_count_issue']}, goals_verb_issue={row['goals_verb_issue']}, "
            f"has_default_roles={row['has_default_roles']}, default_roles_count={row['default_roles_count']}, "
            f"duplicate_sections={row['has_duplicate_sections']}"
        )
        lines.append("")
        lines.append("Proposal / Input:")
        lines.append(row["proposal"].strip())
        lines.append("")
        lines.append("Model Output:")
        lines.append(row["output"].strip())

    export_file.write_text("\n".join(lines), encoding="utf-8")
    print(f"\nSaved inference outputs to: {export_file}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

MODEL 1 INFERENCE TEST - STEP 7 COMPACT STRICT PROMPTS
Quick mode: True
Base strategies: ['B']
Conditional C fallback: True
Adapter path: C:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\qwen_model1_overview_lora_1p5b
Selected dataset rows (1-based): [6, 7, 8, 9, 10]
Examples to run: 5
Show full output: True
Export txt: True
Few-shot examples loaded: 0 (disabled for compact strict prompts)
Strict prompt integration active: True
Strict prompt sections loaded: ['summary', 'roles', 'features', 'goals', 'timeline']
Strict prompt rule lines injected: 8

EXAMPLE 1 (dataset row 6)
Best strategy: B: guided+compact-strict (score=23)
Sections missing: None | Weeks: 4/4 | Status: False | Backlog: False | Proposal: False | Generic timeline: False | Repeated weeks: False | Goals count issue: True | Goals verb issue: True | Has default roles: True | Default roles count: 3 | Duplicate sections: False
Result: FAIL
Output:
Title: SafeSchool Campus Safety System
Summary: Develop a web-based

## Step 6.1: Compare dataset target vs generated output

Pull the same inference dataset rows and compare target responses against fresh model outputs.

In [7]:
# Compare expected dataset responses vs fresh model outputs on fixed rows.
# Run Step 6 first so helper functions and model_infer are loaded.

required_names = ["_generate", "_compliance_score"]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Step 6 first so comparison helpers are loaded. Missing: " + ", ".join(missing)
    )

if "selected_rows_sorted" in globals():
    compare_rows = selected_rows_sorted
else:
    compare_rows = [6, 7, 8, 9, 10]

if "dataset_file" not in globals():
    dataset_file = DATASET_DIR / "model1_description_to_part1.jsonl"
if not dataset_file.exists():
    raise FileNotFoundError(f"Missing dataset for comparison: {dataset_file}")

if "examples_by_row" not in globals():
    examples_by_row = {}
    with open(dataset_file, "r", encoding="utf-8") as f:
        for row_num, line in enumerate(f, start=1):
            if row_num not in set(compare_rows):
                continue
            line = line.strip()
            if line:
                examples_by_row[row_num] = json.loads(line)

missing_rows = [row for row in compare_rows if row not in examples_by_row]
if missing_rows:
    raise ValueError(f"Missing requested rows in dataset: {missing_rows}")

def _normalize_target_for_comparison(response_text: str, proposal_text: str) -> str:
    if "_sanitize_response" not in globals():
        return response_text.strip()
    try:
        return _sanitize_response(response_text, proposal_text).strip()
    except TypeError:
        return _sanitize_response(response_text).strip()

print("=" * 90)
print("MODEL 1 DATASET VS GENERATED COMPARISON")
print("=" * 90)
print(f"Dataset: {dataset_file}")
print(f"Rows: {compare_rows}")

comparison_results = []
for row_num in compare_rows:
    example = examples_by_row[row_num]
    proposal = example.get("prompt", "").strip()
    target_raw = example.get("response", "").strip()

    target_clean = _normalize_target_for_comparison(target_raw, proposal)
    target_score = _compliance_score(target_clean)

    if "GUIDED_PREFIX" in globals():
        prompt_cmp = GUIDED_PREFIX + "Input:\n" + proposal + "\n\nOutput:\n"
    else:
        prompt_cmp = proposal
    generated_raw = _generate(prompt_cmp)
    generated_clean = _normalize_target_for_comparison(generated_raw, proposal)
    generated_score = _compliance_score(generated_clean)

    comparison_results.append({
        "row": row_num,
        "target_score": target_score,
        "generated_score": generated_score,
        "target_output": target_clean,
        "generated_output": generated_clean,
    })

    print(f"\nRow {row_num}")
    print(f"Target score: {target_score['score']} | Generated score: {generated_score['score']}")
    print(f"Target checks: {target_score}")
    print(f"Generated checks: {generated_score}")

avg_target = sum(r["target_score"]["score"] for r in comparison_results) / len(comparison_results)
avg_generated = sum(r["generated_score"]["score"] for r in comparison_results) / len(comparison_results)
print("\n" + "=" * 90)
print(f"Average target score: {avg_target:.2f}")
print(f"Average generated score: {avg_generated:.2f}")
print("=" * 90)

MODEL 1 DATASET VS GENERATED COMPARISON
Dataset: c:\Users\Aaron\GitHub Repos\My-Crew-Manager\AI\llms\fine_tune\dataset\model1_description_to_part1.jsonl
Rows: [6, 7, 8, 9, 10]

Row 6
Target score: 21 | Generated score: 21
Target checks: {'score': 21, 'missing': [], 'weeks': 4, 'has_status': False, 'has_backlog': False, 'has_proposal': False, 'has_generic_timeline': False, 'has_duplicate_sections': False}
Generated checks: {'score': 21, 'missing': [], 'weeks': 4, 'has_status': False, 'has_backlog': False, 'has_proposal': False, 'has_generic_timeline': False, 'has_duplicate_sections': False}

Row 7
Target score: 21 | Generated score: 21
Target checks: {'score': 21, 'missing': [], 'weeks': 4, 'has_status': False, 'has_backlog': False, 'has_proposal': False, 'has_generic_timeline': False, 'has_duplicate_sections': False}
Generated checks: {'score': 21, 'missing': [], 'weeks': 4, 'has_status': False, 'has_backlog': False, 'has_proposal': False, 'has_generic_timeline': False, 'has_duplicate_